# Development of `StateVector` object

In [1]:
import keyword 
import math 
from numbers import Real
import keyword

from dataclasses import dataclass

# 1) Helper Functions

- `_validate_text`
- `_validate_name`
- `_validate_scalar`

## 1.1 - `_validate_text`

In [2]:
def _validate_text(value:object, label:str) -> str:
    """
    Validate and return non-empty text.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    str
        Input text with leading and trailing whitespace removed.

    Example
    -------
    >>> _validate_text("  m/s  ", "StateField unit")
    'm/s'
    """
    if not isinstance(value, str):
        raise TypeError(
            f"{label} must be a string; received {type(value).__name__}"
        )

    cleaned = value.strip()

    if cleaned == "":
        raise ValueError(
            f"{label} must be a non-empty string"
        )

    return cleaned

In [3]:
# Example Usage:
print(_validate_text("123   ", "StateField unit"))
#print(_validate_text(123, "StateField unit"))

123


## 1.2 - `_validate_name`

In [4]:
def _validate_name(value: object, label: str) -> str:
    """
    Validate and return a symbolic name.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    str
        Validated name with leading and trailing whitespace removed.

    Example
    -------
    >>> _validate_name("  x_dot  ", "StateField name")
    'x_dot'
    """
    name = _validate_text(value, label)

    if not name.isidentifier():
        raise ValueError(
            f"{label} must be a valid Python identifier; received {name!r}."
        )

    if keyword.iskeyword(name):
        raise ValueError(
            f"{label} cannot be a Python keyword; received {name!r}."
        )

    return name

In [5]:
# Example Usage:
#_validate_name("m/s", "StateField name")
_validate_name("x_dot", "StateField name")

'x_dot'

## 1.3 - `_validate_scalar`

In [6]:
def _validate_scalar(value: object, label: str) -> float:
    """
    Validate and return a finite real scalar.

    Parameters
    ----------
    value : object
        Input value to validate.

    label : str
        Human-readable name used in error messages.

    Returns
    -------
    float
        Validated scalar converted to float.

    Example
    -------
    >>> _validate_scalar(10, "StateField value")
    10.0
    """
    if isinstance(value, bool):
        raise TypeError(
            f"{label} must be a finite real scalar; boolean values are not allowed."
        )

    if not isinstance(value, Real):
        raise TypeError(
            f"{label} must be a finite real scalar; received {type(value).__name__}."
        )

    scalar = float(value)

    if not math.isfinite(scalar):
        raise ValueError(
            f"{label} must be finite; received {scalar}."
        )

    return scalar

In [7]:
# Example Usage:
#_validate_scalar(True, "x_dot")
_validate_scalar(1, "x_dot")

1.0

# 2) `StateField`

In [8]:
@dataclass(frozen=True)
class StateField:
    """
    To insantiate One scalar component of a state vector in the StateVector class.

    Parameters
    ----------
    name : str
        Symbolic name of the state component.

    value : float
        Numerical value of the state component.

    unit : str
        Unit associated with the state component.

    Usage
    -----
    >>> StateField("x", 10.0, "m")
    StateField(name='x', value=10.0, unit='m')
    """

    name: str
    value: float
    unit: str

    def __post_init__(self) -> None:
        name = _validate_name(self.name, "StateField name")

        value = _validate_scalar(
            self.value,
            f"Value for state field {name!r}",
        )

        unit = _validate_text(
            self.unit,
            f"Unit for state field {name!r}",
        )

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "value", value)
        object.__setattr__(self, "unit", unit)

In [ ]:
# Example Usage:
field = StateField("  x  ", 10, "  m  ")

print(field)
print(field.name)
print(field.value)
print(field.unit)


StateField(name='x', value=10.0, unit='m')
x
10.0
m
